In [1]:
import json
from pathlib import Path

path = Path("dataset/teacher_subset_100sat_100unsat_matched_sft_noleak.jsonl")

rows = []
with path.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

In [7]:
# !uv pip install pandas

In [16]:
import pandas as pd

df = pd.DataFrame(rows)
df.head().iloc[0]["messages"][0]

{'role': 'system',
 'content': 'You are an expert in propositional logic and Boolean satisfiability (SAT).\nYou are solving SATBench-style natural-language logic puzzles.\n\nImportant reasoning rules:\n- Use only the constraints stated in the conditions and formal CNF information.\n- The scenario is background only and adds no hidden constraints.\n- Treat all variables as independent Boolean decisions unless the conditions explicitly state otherwise.\n- Do not add commonsense assumptions such as mutual exclusivity, exactly-one constraints, or real-world causal links unless they are stated in the conditions.\n- Variables not mentioned in the conditions are irrelevant to satisfiability and may be assigned arbitrarily.\n\nYour task:\n1. Decide whether the puzzle is SAT or UNSAT.\n2. If SAT, give one satisfying assignment or enough assignment information to verify the clauses.\n3. If UNSAT, identify a contradiction or an UNSAT core/relevant conflicting clauses.\n4. Explain the reasoning br

In [18]:
import json
import random
from pathlib import Path
from collections import Counter

# =========================
# Configuration
# =========================

INPUT_PATH = Path("dataset/teacher_subset_100sat_100unsat_matched_sft_noleak.jsonl")

OUT_DIR = Path("dataset/sft_by_label")

# For 100 SAT + 100 UNSAT, this gives:
# SAT:   80 train, 20 test
# UNSAT: 80 train, 20 test
TEST_PER_LABEL = 20

RANDOM_SEED = 42


# =========================
# Helper functions
# =========================

def load_jsonl(path):
    """Load a JSONL file into a list of Python dictionaries."""
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_no}: {e}") from e
    return rows


def write_jsonl(rows, path):
    """Write a list of dictionaries to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def get_label(row):
    """
    Get SAT/UNSAT label from one row.

    The cleaned SFT file should have:
        row["ground_truth_label"] == "SAT" or "UNSAT"

    This function also includes fallbacks in case your file structure changes.
    """
    label = row.get("ground_truth_label")

    if label is None and isinstance(row.get("ground_truth"), dict):
        label = row["ground_truth"].get("label")

    if label is None and isinstance(row.get("ground_truth"), dict):
        satisfiable = row["ground_truth"].get("satisfiable")
        if satisfiable is True:
            label = "SAT"
        elif satisfiable is False:
            label = "UNSAT"

    if label is None:
        raise ValueError(f"Could not find label for row with keys: {list(row.keys())}")

    label = str(label).strip().upper()
    if label not in {"SAT", "UNSAT"}:
        raise ValueError(f"Unexpected label: {label}")

    return label


# =========================
# Load and split
# =========================

rows = load_jsonl(INPUT_PATH)
print(f"Loaded {len(rows)} rows from {INPUT_PATH}")

# Group by SAT / UNSAT
by_label = {"SAT": [], "UNSAT": []}

for row in rows:
    label = get_label(row)
    by_label[label].append(row)

print("Original counts:")
print({label: len(items) for label, items in by_label.items()})

# Basic validation
if len(by_label["SAT"]) == 0 or len(by_label["UNSAT"]) == 0:
    raise ValueError("Both SAT and UNSAT groups must be non-empty.")

if TEST_PER_LABEL >= len(by_label["SAT"]):
    raise ValueError("TEST_PER_LABEL is too large for SAT samples.")

if TEST_PER_LABEL >= len(by_label["UNSAT"]):
    raise ValueError("TEST_PER_LABEL is too large for UNSAT samples.")


# Shuffle each label separately and split
rng = random.Random(RANDOM_SEED)

splits = {}

for label in ["SAT", "UNSAT"]:
    label_rows = list(by_label[label])
    rng.shuffle(label_rows)

    test_rows = label_rows[:TEST_PER_LABEL]
    train_rows = label_rows[TEST_PER_LABEL:]

    splits[label] = {
        "train": train_rows,
        "test": test_rows,
    }


# =========================
# Save files
# =========================

write_jsonl(splits["SAT"]["train"], OUT_DIR / "sat" / "train.jsonl")
write_jsonl(splits["SAT"]["test"], OUT_DIR / "sat" / "test.jsonl")

write_jsonl(splits["UNSAT"]["train"], OUT_DIR / "unsat" / "train.jsonl")
write_jsonl(splits["UNSAT"]["test"], OUT_DIR / "unsat" / "test.jsonl")


# Save a small split report
report = {
    "input_path": str(INPUT_PATH),
    "output_dir": str(OUT_DIR),
    "random_seed": RANDOM_SEED,
    "test_per_label": TEST_PER_LABEL,
    "counts": {
        "sat": {
            "train": len(splits["SAT"]["train"]),
            "test": len(splits["SAT"]["test"]),
            "total": len(by_label["SAT"]),
        },
        "unsat": {
            "train": len(splits["UNSAT"]["train"]),
            "test": len(splits["UNSAT"]["test"]),
            "total": len(by_label["UNSAT"]),
        },
        "all": {
            "train": len(splits["SAT"]["train"]) + len(splits["UNSAT"]["train"]),
            "test": len(splits["SAT"]["test"]) + len(splits["UNSAT"]["test"]),
            "total": len(rows),
        },
    },
}

report_path = OUT_DIR / "split_report.json"
report_path.parent.mkdir(parents=True, exist_ok=True)

with report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)


# =========================
# Print summary
# =========================

print("\nSaved files:")
print(OUT_DIR / "sat" / "train.jsonl")
print(OUT_DIR / "sat" / "test.jsonl")
print(OUT_DIR / "unsat" / "train.jsonl")
print(OUT_DIR / "unsat" / "test.jsonl")
print(report_path)

print("\nSplit report:")
print(json.dumps(report, indent=2))

Loaded 200 rows from dataset/teacher_subset_100sat_100unsat_matched_sft_noleak.jsonl
Original counts:
{'SAT': 100, 'UNSAT': 100}

Saved files:
dataset/sft_by_label/sat/train.jsonl
dataset/sft_by_label/sat/test.jsonl
dataset/sft_by_label/unsat/train.jsonl
dataset/sft_by_label/unsat/test.jsonl
dataset/sft_by_label/split_report.json

Split report:
{
  "input_path": "dataset/teacher_subset_100sat_100unsat_matched_sft_noleak.jsonl",
  "output_dir": "dataset/sft_by_label",
  "random_seed": 42,
  "test_per_label": 20,
  "counts": {
    "sat": {
      "train": 80,
      "test": 20,
      "total": 100
    },
    "unsat": {
      "train": 80,
      "test": 20,
      "total": 100
    },
    "all": {
      "train": 160,
      "test": 40,
      "total": 200
    }
  }
}


In [3]:
import json
import re
import time
import html
from pathlib import Path
from threading import Thread
from collections import Counter

import torch
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display, HTML

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextIteratorStreamer,
)


# ============================================================
# 1. Configuration
# ============================================================

MODEL_ID = "Qwen/Qwen3.5-0.8B-Base"

SAT_TEST_PATH = Path("dataset/sft_by_label/sat/test.jsonl")
UNSAT_TEST_PATH = Path("dataset/sft_by_label/unsat/test.jsonl")

OUTPUT_DIR = Path("results/baseline_qwen35_08b_base_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSONL = OUTPUT_DIR / "qwen35_08b_base_test_predictions.jsonl"
OUTPUT_CSV = OUTPUT_DIR / "qwen35_08b_base_test_predictions.csv"

# Set this to True if the model is already downloaded and you are running offline on Adroit.
LOCAL_FILES_ONLY = False

# Generation settings.
# For a base model, deterministic decoding is usually better for evaluation.
MAX_NEW_TOKENS = 8192
DO_SAMPLE = False
TEMPERATURE = 0.0
TOP_P = 1.0

# Streaming display settings.
# Set to False if you only want tqdm progress and no streamed text.
STREAM_OUTPUT = True

# If you only want to smoke test a few examples first, set this to an integer.
# Example: LIMIT_PER_SPLIT = 2
LIMIT_PER_SPLIT = None


# ============================================================
# 2. JSONL loading helpers
# ============================================================

def load_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON in {path} line {line_no}: {e}") from e
    return rows


def write_jsonl(rows, path):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


sat_rows = load_jsonl(SAT_TEST_PATH)
unsat_rows = load_jsonl(UNSAT_TEST_PATH)

if LIMIT_PER_SPLIT is not None:
    sat_rows = sat_rows[:LIMIT_PER_SPLIT]
    unsat_rows = unsat_rows[:LIMIT_PER_SPLIT]

# Add explicit split labels for easier reporting.
for row in sat_rows:
    row["_eval_split"] = "sat_test"
    row["_expected_label_from_file"] = "SAT"

for row in unsat_rows:
    row["_eval_split"] = "unsat_test"
    row["_expected_label_from_file"] = "UNSAT"

test_rows = sat_rows + unsat_rows

print(f"Loaded SAT test rows:   {len(sat_rows)}")
print(f"Loaded UNSAT test rows: {len(unsat_rows)}")
print(f"Loaded total rows:      {len(test_rows)}")


# ============================================================
# 3. Label and prompt helpers
# ============================================================

def get_ground_truth_label(row):
    """
    Prefer explicit ground_truth_label if present.
    Fall back to ground_truth.label, ground_truth.satisfiable,
    or the file label we attached above.
    """
    label = row.get("ground_truth_label")

    if label is None and isinstance(row.get("ground_truth"), dict):
        label = row["ground_truth"].get("label")

    if label is None and isinstance(row.get("ground_truth"), dict):
        satisfiable = row["ground_truth"].get("satisfiable")
        if satisfiable is True:
            label = "SAT"
        elif satisfiable is False:
            label = "UNSAT"

    if label is None:
        label = row.get("_expected_label_from_file")

    if label is None:
        raise ValueError(f"Cannot find ground-truth label. Row keys: {list(row.keys())}")

    label = str(label).strip().upper()
    if label not in {"SAT", "UNSAT"}:
        raise ValueError(f"Unexpected ground-truth label: {label}")

    return label


def get_prompt_messages(row):
    """
    SFT rows usually look like:
      {
        "messages": [
          {"role": "system", "content": "..."},
          {"role": "user", "content": "..."},
          {"role": "assistant", "content": "...teacher answer..."}
        ]
      }

    For evaluation, we must remove the assistant target.
    """
    messages = row.get("messages")
    if not isinstance(messages, list):
        raise ValueError(f"Row does not contain a valid messages list. Keys: {list(row.keys())}")

    prompt_messages = []
    for msg in messages:
        role = msg.get("role")
        content = msg.get("content", "")
        if role == "assistant":
            continue
        prompt_messages.append({"role": role, "content": content})

    if not prompt_messages:
        raise ValueError("No non-assistant messages found.")

    return prompt_messages


def messages_to_prompt(tokenizer, messages):
    """
    Use the tokenizer chat template if available.
    For a Base model, the tokenizer may not have a useful chat template,
    so we fall back to a simple plain-text format.
    """
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        parts = []
        for msg in messages:
            role = msg.get("role", "user").upper()
            content = msg.get("content", "")
            parts.append(f"{role}:\n{content}")
        parts.append("ASSISTANT:\n")
        return "\n\n".join(parts)


def extract_predicted_label(text):
    """
    Extract final SAT/UNSAT prediction.

    First looks for bracketed labels like [SAT] or [UNSAT].
    If not found, falls back to the last occurrence of the words SAT/UNSAT.
    UNSAT is checked carefully so it is not confused with SAT.
    """
    if not text:
        return None

    bracket_matches = re.findall(r"\[\s*(UNSAT|SAT)\s*\]", text, flags=re.IGNORECASE)
    if bracket_matches:
        return bracket_matches[-1].upper()

    word_matches = re.findall(r"\b(UNSAT|SAT)\b", text, flags=re.IGNORECASE)
    if word_matches:
        return word_matches[-1].upper()

    return None


# ============================================================
# 4. Load tokenizer and model
# ============================================================

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    local_files_only=LOCAL_FILES_ONLY,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model: {MODEL_ID}")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
    local_files_only=LOCAL_FILES_ONLY,
)

model.eval()

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ============================================================
# 5. Streaming generation helper
# ============================================================

@torch.no_grad()
def generate_one(row, row_id):
    """
    Generate one model response with optional streaming display.
    Returns a result dictionary.
    """
    gold = get_ground_truth_label(row)
    messages = get_prompt_messages(row)
    prompt_text = messages_to_prompt(tokenizer, messages)

    inputs = tokenizer(prompt_text, return_tensors="pt")

    # Put input tensors on the same device as the embedding layer.
    # This works with device_map="auto" too.
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    if DO_SAMPLE:
        generation_kwargs.update(
            temperature=TEMPERATURE,
            top_p=TOP_P,
        )

    start_time = time.time()

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    generated_text = ""

    if STREAM_OUTPUT:
        header = (
            f"<b>Example {row_id}</b> | "
            f"split={row.get('_eval_split')} | "
            f"gold={gold}"
        )
        display_handle = display(
            HTML(header + "<pre style='white-space: pre-wrap; font-size: 13px;'></pre>"),
            display_id=True,
        )

    for new_text in streamer:
        generated_text += new_text

        if STREAM_OUTPUT:
            safe_text = html.escape(generated_text)
            display_handle.update(
                HTML(
                    header
                    + "<pre style='white-space: pre-wrap; font-size: 13px;'>"
                    + safe_text
                    + "</pre>"
                )
            )

    thread.join()
    elapsed = time.time() - start_time

    pred = extract_predicted_label(generated_text)
    correct = pred == gold

    input_tokens = int(inputs["input_ids"].shape[-1])
    output_tokens = len(tokenizer.encode(generated_text, add_special_tokens=False))

    result = {
        "row_id": row_id,
        "eval_split": row.get("_eval_split"),
        "gold_label": gold,
        "predicted_label": pred,
        "correct": correct,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "seconds": elapsed,
        "tokens_per_second": output_tokens / elapsed if elapsed > 0 else None,
        "prompt_text": prompt_text,
        "model_response": generated_text,
    }

    return result


# ============================================================
# 6. Run evaluation with tqdm progress
# ============================================================

results = []

for i, row in enumerate(tqdm(test_rows, desc="Evaluating Qwen3.5-0.8B-Base")):
    try:
        result = generate_one(row, row_id=i)
    except Exception as e:
        result = {
            "row_id": i,
            "eval_split": row.get("_eval_split"),
            "gold_label": get_ground_truth_label(row),
            "predicted_label": None,
            "correct": False,
            "error": repr(e),
            "model_response": "",
        }
        print(f"Error on row {i}: {repr(e)}")

    results.append(result)

    # Save after every example so you do not lose progress if the notebook stops.
    write_jsonl(results, OUTPUT_JSONL)


print(f"Saved raw predictions to: {OUTPUT_JSONL}")


# ============================================================
# 7. Summarize results
# ============================================================

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)

def accuracy_for(sub_df):
    if len(sub_df) == 0:
        return None
    return float(sub_df["correct"].mean())

summary = {
    "model_id": MODEL_ID,
    "num_total": len(df),
    "num_sat_test": int((df["eval_split"] == "sat_test").sum()),
    "num_unsat_test": int((df["eval_split"] == "unsat_test").sum()),
    "overall_accuracy": accuracy_for(df),
    "sat_test_accuracy": accuracy_for(df[df["eval_split"] == "sat_test"]),
    "unsat_test_accuracy": accuracy_for(df[df["eval_split"] == "unsat_test"]),
    "predicted_label_counts": dict(Counter(df["predicted_label"].fillna("NONE"))),
    "gold_label_counts": dict(Counter(df["gold_label"].fillna("NONE"))),
    "avg_output_tokens": float(df["output_tokens"].mean()) if "output_tokens" in df else None,
    "avg_tokens_per_second": float(df["tokens_per_second"].dropna().mean()) if "tokens_per_second" in df else None,
}

summary_path = OUTPUT_DIR / "qwen35_08b_base_test_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nSummary:")
print(json.dumps(summary, indent=2))

print(f"\nSaved CSV to: {OUTPUT_CSV}")
print(f"Saved summary to: {summary_path}")

display(df[[
    "row_id",
    "eval_split",
    "gold_label",
    "predicted_label",
    "correct",
    "output_tokens",
    "seconds",
    "tokens_per_second",
]].head())

Loaded SAT test rows:   20
Loaded UNSAT test rows: 20
Loaded total rows:      40
Loading tokenizer: Qwen/Qwen3.5-0.8B-Base
Loading model: Qwen/Qwen3.5-0.8B-Base


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

CUDA available: True
GPU: NVIDIA A100 80GB PCIe


Evaluating Qwen3.5-0.8B-Base:   0%|          | 0/40 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [12]:
# ============================================================
# Full-parameter SFT for Qwen/Qwen3.5-0.8B-Base
#
# Train data:
#   dataset/sft_by_label/sat/train.jsonl
#   dataset/sft_by_label/unsat/train.jsonl
#
# Output:
#   results/sft_qwen35_08b_base_sat_unsat_full/
#
# Supports automatic resumption from latest checkpoint.
# Compatible with newer transformers versions where Trainer(tokenizer=...)
# is removed.
# ============================================================

import os
import json
import random
import inspect
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
from torch.utils.data import Dataset

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from transformers.trainer_utils import get_last_checkpoint


# ============================================================
# 1. Configuration
# ============================================================

MODEL_ID = "Qwen/Qwen3.5-0.8B-Base"

SAT_TRAIN_PATH = Path("dataset/sft_by_label/sat/train.jsonl")
UNSAT_TRAIN_PATH = Path("dataset/sft_by_label/unsat/train.jsonl")

OUTPUT_DIR = Path("results/sft_qwen35_08b_base_sat_unsat_full")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_DIR = OUTPUT_DIR / "final_model"

# Set this to True if the model is already downloaded and you are running offline.
LOCAL_FILES_ONLY = False

# For a quick notebook smoke test, set this to 2 or 4.
# For full training, set it to None.
LIMIT_PER_LABEL = None

RANDOM_SEED = 42

# Full-parameter fine-tuning settings.
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03

# If CUDA OOM happens, reduce to 2048.
MAX_LENGTH = 4096

LOGGING_STEPS = 1
SAVE_STEPS = 10
SAVE_TOTAL_LIMIT = 3

# If True, ignore existing checkpoints and start from base model.
# This does not delete old checkpoints.
START_FROM_SCRATCH = False


# ============================================================
# 2. Reproducibility and environment check
# ============================================================

print("transformers version:", transformers.__version__)

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA device count:", torch.cuda.device_count())
    print("bf16 supported:", torch.cuda.is_bf16_supported())


# ============================================================
# 3. Load JSONL files
# ============================================================

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON in {path} line {line_no}: {e}") from e

    return rows


sat_rows = load_jsonl(SAT_TRAIN_PATH)
unsat_rows = load_jsonl(UNSAT_TRAIN_PATH)

if LIMIT_PER_LABEL is not None:
    sat_rows = sat_rows[:LIMIT_PER_LABEL]
    unsat_rows = unsat_rows[:LIMIT_PER_LABEL]

for row in sat_rows:
    row["_train_label"] = "SAT"

for row in unsat_rows:
    row["_train_label"] = "UNSAT"

train_rows = sat_rows + unsat_rows
random.shuffle(train_rows)

print(f"SAT train rows:   {len(sat_rows)}")
print(f"UNSAT train rows: {len(unsat_rows)}")
print(f"Total train rows: {len(train_rows)}")

if len(train_rows) == 0:
    raise ValueError("No training rows loaded. Check your train JSONL paths.")


# ============================================================
# 4. Message formatting helpers
# ============================================================

def split_messages(row: Dict[str, Any]):
    """
    Expected SFT row format:

    {
      "messages": [
        {"role": "system", "content": "..."},
        {"role": "user", "content": "..."},
        {"role": "assistant", "content": "...teacher answer..."}
      ]
    }

    We train only on the assistant response.
    System/user prompt tokens are masked with label = -100.
    """

    messages = row.get("messages")

    if not isinstance(messages, list):
        raise ValueError(f"Row has no valid messages field. Keys: {list(row.keys())}")

    prompt_messages = []
    assistant_text = None

    for msg in messages:
        role = msg.get("role")
        content = msg.get("content", "")

        if role == "assistant":
            assistant_text = content
        else:
            prompt_messages.append({
                "role": role,
                "content": content,
            })

    if assistant_text is None:
        raise ValueError("No assistant message found in row.")

    if len(prompt_messages) == 0:
        raise ValueError("No prompt messages found before assistant message.")

    return prompt_messages, assistant_text


def format_prompt_fallback(messages: List[Dict[str, str]]) -> str:
    """
    Fallback prompt format for base models if tokenizer chat template is unavailable.
    """

    parts = []

    for msg in messages:
        role = msg.get("role", "user")
        content = msg.get("content", "")

        if role == "system":
            parts.append(f"### System:\n{content}")
        elif role == "user":
            parts.append(f"### User:\n{content}")
        else:
            parts.append(f"### {role.capitalize()}:\n{content}")

    parts.append("### Assistant:\n")
    return "\n\n".join(parts)


def build_prompt(tokenizer, prompt_messages: List[Dict[str, str]]) -> str:
    """
    Prefer tokenizer chat template.
    Fall back to a plain-text instruction format if unavailable.
    """

    try:
        return tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return format_prompt_fallback(prompt_messages)


# ============================================================
# 5. Load tokenizer
# ============================================================

print(f"Loading tokenizer: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    local_files_only=LOCAL_FILES_ONLY,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token)
print("eos_token_id:", tokenizer.eos_token_id)


# ============================================================
# 6. Dataset with assistant-only loss masking
# ============================================================

class ChatSFTDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length: int):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        prompt_messages, assistant_text = split_messages(row)
        prompt_text = build_prompt(self.tokenizer, prompt_messages)

        # Add EOS to teach the model where to stop.
        assistant_text_with_eos = assistant_text + self.tokenizer.eos_token

        prompt_ids = self.tokenizer(
            prompt_text,
            add_special_tokens=False,
        )["input_ids"]

        assistant_ids = self.tokenizer(
            assistant_text_with_eos,
            add_special_tokens=False,
        )["input_ids"]

        # Keep room for at least part of the assistant answer.
        # If prompt + assistant exceeds MAX_LENGTH, truncate the assistant first.
        available_for_assistant = self.max_length - len(prompt_ids)

        if available_for_assistant <= 0:
            # Prompt alone is too long. Keep the last MAX_LENGTH - 1 prompt tokens
            # and one assistant token so the sample remains trainable.
            prompt_ids = prompt_ids[-(self.max_length - 1):]
            assistant_ids = assistant_ids[:1]
        else:
            assistant_ids = assistant_ids[:available_for_assistant]

        input_ids = prompt_ids + assistant_ids
        labels = [-100] * len(prompt_ids) + assistant_ids.copy()
        attention_mask = [1] * len(input_ids)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


@dataclass
class DataCollatorForCausalLMWithPadding:
    tokenizer: Any
    label_pad_token_id: int = -100

    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)
        pad_id = self.tokenizer.pad_token_id

        input_ids = []
        attention_mask = []
        labels = []

        for x in features:
            cur_len = len(x["input_ids"])
            pad_len = max_len - cur_len

            input_ids.append(x["input_ids"] + [pad_id] * pad_len)
            attention_mask.append(x["attention_mask"] + [0] * pad_len)
            labels.append(x["labels"] + [self.label_pad_token_id] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


train_dataset = ChatSFTDataset(
    rows=train_rows,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

collator = DataCollatorForCausalLMWithPadding(tokenizer=tokenizer)

print("\nDataset sanity check:")
zero_trainable_count = 0

for i in range(len(train_dataset)):
    sample = train_dataset[i]
    total_tokens = len(sample["input_ids"])
    trainable_tokens = sum(x != -100 for x in sample["labels"])
    masked_tokens = sum(x == -100 for x in sample["labels"])

    if trainable_tokens == 0:
        zero_trainable_count += 1

    if i < 3:
        print(
            f"sample={i}, "
            f"total_tokens={total_tokens}, "
            f"masked_prompt_tokens={masked_tokens}, "
            f"trainable_assistant_tokens={trainable_tokens}"
        )

print(f"Rows with 0 trainable assistant tokens: {zero_trainable_count}")

if zero_trainable_count > 0:
    raise ValueError(
        "Some rows have 0 trainable assistant tokens. "
        "Increase MAX_LENGTH or shorten prompts."
    )


# ============================================================
# 7. Load model for full-parameter fine-tuning
# ============================================================

def choose_dtype():
    if not torch.cuda.is_available():
        return torch.float32

    if torch.cuda.is_bf16_supported():
        return torch.bfloat16

    return torch.float16


dtype = choose_dtype()

print(f"\nLoading model for full-parameter SFT: {MODEL_ID}")
print("Using dtype:", dtype)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    trust_remote_code=True,
    local_files_only=LOCAL_FILES_ONLY,
)

# Important for training.
model.config.use_cache = False

# Saves activation memory.
model.gradient_checkpointing_enable()

# For gradient checkpointing compatibility.
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# Full fine-tuning: make sure all parameters are trainable.
for param in model.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.2f}%")


# ============================================================
# 8. Version-compatible TrainingArguments
# ============================================================

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16


def make_training_args(**kwargs):
    """
    Drop unsupported TrainingArguments kwargs automatically.
    This avoids errors across different transformers versions.
    """
    valid_args = set(inspect.signature(TrainingArguments.__init__).parameters.keys())

    filtered = {}
    dropped = {}

    for key, value in kwargs.items():
        if key in valid_args:
            filtered[key] = value
        else:
            dropped[key] = value

    if dropped:
        print("\nDropped unsupported TrainingArguments keys:")
        for key in dropped:
            print(f"  - {key}")

    return TrainingArguments(**filtered)


training_args = make_training_args(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,

    bf16=use_bf16,
    fp16=use_fp16,

    optim="adamw_torch",

    report_to="none",
    remove_unused_columns=False,

    seed=RANDOM_SEED,
    data_seed=RANDOM_SEED,

    save_safetensors=True,
)


# ============================================================
# 9. Version-compatible Trainer construction
# ============================================================

def make_trainer(model, args, train_dataset, data_collator, tokenizer):
    """
    Newer transformers versions removed Trainer(tokenizer=...).
    Some versions use processing_class=... instead.
    This helper passes only supported kwargs.
    """

    trainer_kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_dataset,
        "data_collator": data_collator,
    }

    valid_trainer_args = set(inspect.signature(Trainer.__init__).parameters.keys())

    if "processing_class" in valid_trainer_args:
        trainer_kwargs["processing_class"] = tokenizer
        print("Trainer will use processing_class=tokenizer")
    elif "tokenizer" in valid_trainer_args:
        trainer_kwargs["tokenizer"] = tokenizer
        print("Trainer will use tokenizer=tokenizer")
    else:
        print("Trainer supports neither tokenizer nor processing_class. Omitting tokenizer argument.")

    filtered_kwargs = {
        key: value for key, value in trainer_kwargs.items()
        if key in valid_trainer_args
    }

    dropped = set(trainer_kwargs.keys()) - set(filtered_kwargs.keys())
    if dropped:
        print("Dropped unsupported Trainer keys:")
        for key in dropped:
            print(f"  - {key}")

    return Trainer(**filtered_kwargs)


trainer = make_trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
    tokenizer=tokenizer,
)


# ============================================================
# 10. Resume from latest checkpoint if it exists
# ============================================================

last_checkpoint = None

if OUTPUT_DIR.exists() and not START_FROM_SCRATCH:
    last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))

if last_checkpoint is not None:
    print(f"\nFound checkpoint. Resuming from: {last_checkpoint}")
else:
    print("\nNo checkpoint found. Starting full SFT from the base model.")


# ============================================================
# 11. Train
# ============================================================

train_result = trainer.train(resume_from_checkpoint=last_checkpoint)

print("\nTraining finished.")
print(train_result)


# ============================================================
# 12. Save final full fine-tuned model and tokenizer
# ============================================================

FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

metrics = train_result.metrics
metrics_path = OUTPUT_DIR / "train_metrics.json"

with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"\nSaved final full fine-tuned model to: {FINAL_MODEL_DIR}")
print(f"Saved tokenizer to: {FINAL_MODEL_DIR}")
print(f"Saved training metrics to: {metrics_path}")
print(f"Training checkpoints are in: {OUTPUT_DIR}")

transformers version: 5.7.0
CUDA available: True
GPU: NVIDIA A100 80GB PCIe
CUDA device count: 2
bf16 supported: True
SAT train rows:   80
UNSAT train rows: 80
Total train rows: 160
Loading tokenizer: Qwen/Qwen3.5-0.8B-Base
pad_token: <|endoftext|>
pad_token_id: 248044
eos_token: <|endoftext|>
eos_token_id: 248044

Dataset sanity check:
sample=0, total_tokens=2702, masked_prompt_tokens=971, trainable_assistant_tokens=1731
sample=1, total_tokens=4096, masked_prompt_tokens=1045, trainable_assistant_tokens=3051
sample=2, total_tokens=3351, masked_prompt_tokens=1153, trainable_assistant_tokens=2198
Rows with 0 trainable assistant tokens: 0

Loading model for full-parameter SFT: Qwen/Qwen3.5-0.8B-Base
Using dtype: torch.bfloat16


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainable parameters: 752,393,024
Total parameters:     752,393,024
Trainable percentage: 100.00%

Dropped unsupported TrainingArguments keys:
  - save_safetensors
Trainer will use processing_class=tokenizer


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 248044}.



No checkpoint found. Starting full SFT from the base model.


Step,Training Loss


KeyboardInterrupt: 

In [11]:
import transformers

transformers.__version__

'5.7.0'

In [35]:
import json
from pathlib import Path

path = Path("dataset/sft_by_label/sat/train.jsonl")

rows = []
with path.open() as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

In [36]:
rows[0]

{'messages': [{'role': 'system',
   'content': 'You are an expert in propositional logic and Boolean satisfiability (SAT).\nYou are solving SATBench-style natural-language logic puzzles.\n\nImportant reasoning rules:\n- Use only the constraints stated in the conditions and formal CNF information.\n- The scenario is background only and adds no hidden constraints.\n- Treat all variables as independent Boolean decisions unless the conditions explicitly state otherwise.\n- Do not add commonsense assumptions such as mutual exclusivity, exactly-one constraints, or real-world causal links unless they are stated in the conditions.\n- Variables not mentioned in the conditions are irrelevant to satisfiability and may be assigned arbitrarily.\n\nYour task:\n1. Decide whether the puzzle is SAT or UNSAT.\n2. If SAT, give one satisfying assignment or enough assignment information to verify the clauses.\n3. If UNSAT, identify a contradiction or an UNSAT core/relevant conflicting clauses.\n4. Explain 

In [38]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd

# Change this if your output folder is different
RESULT_DIR = Path("results/eval_sft_qwen35_08b_full_on_test")

PRED_JSONL = RESULT_DIR / "qwen35_08b_base_test_predictions.jsonl"
CORRECTED_JSONL = RESULT_DIR / "qwen35_08b_base_test_predictions_corrected.jsonl"
CORRECTED_CSV = RESULT_DIR / "qwen35_08b_base_test_predictions_corrected.csv"
CORRECTED_SUMMARY = RESULT_DIR / "qwen35_08b_base_test_summary_corrected.json"

# ------------------------------------------------------------
# 1. Load predictions
# ------------------------------------------------------------

rows = []
with PRED_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

print("Loaded rows:", len(rows))

# ------------------------------------------------------------
# 2. Inspect rows with missing prediction
# ------------------------------------------------------------

none_rows = [r for r in rows if r.get("predicted_label") in [None, "", "NONE"]]

print("Rows with missing predicted_label:", len(none_rows))

for r in none_rows:
    print("=" * 100)
    print("dataset_id:", r.get("dataset_id"))
    print("row_id:", r.get("row_id"))
    print("eval_split:", r.get("eval_split"))
    print("gold_label:", r.get("gold_label"))
    print("current predicted_label:", r.get("predicted_label"))
    print("model_response preview:")
    print((r.get("model_response") or ""))

Loaded rows: 40
Rows with missing predicted_label: 1
dataset_id: sat_test_00000
row_id: 0
eval_split: sat_test
gold_label: SAT
current predicted_label: None
model_response preview:
### Step-by-Step Reasoning

**1. Analyze the Problem Structure**
The problem asks whether there exists an assignment of truth values to the variables $x(i, j)$ (where $i \in \{0, 1, 2\}$ and $j \in \{0, 1, 2, 4\}$) that satisfies all six logical conditions provided in the text.

**2. Translate Conditions into Logical Formulas**
Based on the `<readable_cnf>` section of the provided data, we can map the natural language conditions directly to Boolean expressions:
*   **Condition 1:** $\neg x(1, 2) \lor x(1, 1)$
*   **Condition 2:** $\neg x(1, 1) \lor x(2, 1)$
*   **Condition 3:** $\neg x(1, 2) \lor \neg x(2, 1)$
*   **Condition 4:** $x(1, 1) \lor \neg x(2, 1)$
*   **Condition 5:** $x(2, 4) \lor \neg x(0, 0) \lor \neg x(1, 0)$
*   **Condition 6:** $x(1, 2) \lor \neg x(1, 1)$

**3. Evaluate Satisfiability via Z3

In [39]:
# ------------------------------------------------------------
# 3. Manual corrections
# ------------------------------------------------------------
# Use dataset_id if available. Usually it looks like:
#   sat_test_00000
#   unsat_test_00000
#
# You can add more corrections here if needed.

manual_corrections = {
    "sat_test_00000": "SAT",
    # "unsat_test_00003": "UNSAT",
}

for r in rows:
    dataset_id = r.get("dataset_id")
    if dataset_id in manual_corrections:
        new_label = manual_corrections[dataset_id].upper()
        assert new_label in {"SAT", "UNSAT"}
        r["predicted_label"] = new_label
        r["correct"] = (new_label == r.get("gold_label"))
        r["manual_label_correction"] = True
    else:
        r["manual_label_correction"] = False

print("Corrections applied.")

Corrections applied.


In [40]:
# ------------------------------------------------------------
# 4. Save corrected JSONL
# ------------------------------------------------------------

with CORRECTED_JSONL.open("w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# ------------------------------------------------------------
# 5. Save corrected CSV
# ------------------------------------------------------------

df = pd.DataFrame(rows)
df.to_csv(CORRECTED_CSV, index=False)

# ------------------------------------------------------------
# 6. Recompute summary
# ------------------------------------------------------------

def accuracy_for(sub_df):
    if len(sub_df) == 0:
        return None
    return float(sub_df["correct"].mean())

summary = {
    "num_total": len(df),
    "num_sat_test": int((df["eval_split"] == "sat_test").sum()),
    "num_unsat_test": int((df["eval_split"] == "unsat_test").sum()),
    "overall_accuracy": accuracy_for(df),
    "sat_test_accuracy": accuracy_for(df[df["eval_split"] == "sat_test"]),
    "unsat_test_accuracy": accuracy_for(df[df["eval_split"] == "unsat_test"]),
    "predicted_label_counts": dict(Counter(df["predicted_label"].fillna("NONE"))),
    "gold_label_counts": dict(Counter(df["gold_label"].fillna("NONE"))),
    "num_manual_corrections": int(df["manual_label_correction"].sum()),
}

with CORRECTED_SUMMARY.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved corrected files:")
print(CORRECTED_JSONL)
print(CORRECTED_CSV)
print(CORRECTED_SUMMARY)

print("\nCorrected summary:")
print(json.dumps(summary, indent=2))

Saved corrected files:
results/eval_sft_qwen35_08b_full_on_test/qwen35_08b_base_test_predictions_corrected.jsonl
results/eval_sft_qwen35_08b_full_on_test/qwen35_08b_base_test_predictions_corrected.csv
results/eval_sft_qwen35_08b_full_on_test/qwen35_08b_base_test_summary_corrected.json

Corrected summary:
{
  "num_total": 40,
  "num_sat_test": 20,
  "num_unsat_test": 20,
  "overall_accuracy": 0.675,
  "sat_test_accuracy": 0.4,
  "unsat_test_accuracy": 0.95,
  "predicted_label_counts": {
    "SAT": 9,
    "UNSAT": 31
  },
  "gold_label_counts": {
    "SAT": 20,
    "UNSAT": 20
  },
  "num_manual_corrections": 1
}


In [41]:
from transformers import AutoModel

model = AutoModel.from_pretrained("scratch/network/yd1202/COS598B-project/results/sft_qwen35_08b_base_sat_unsat_full/final_model")
model.save_pretrained(push_to_hub=True, repo_name="duanyang25/Finetuned-598B-Qwen3.5-0.8B")

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'scratch/network/yd1202/COS598B-project/results/sft_qwen35_08b_base_sat_unsat_full/final_model'. Use `repo_type` argument if needed.